# CocoFarm C04 – Grey leaf spot lesion segmentation (YOLOv8-seg)

Trains a lesion segmentation model on the Roboflow export, measures severity as
**lesion area ÷ leaf area** and maps it onto the Table 3 rubric (0 / ≤5 / 5–20 / 20–50 / >50 %).

**Before running:** Runtime → Change runtime type → **T4 GPU**.

> Version 3 labels wrap whole leaflets, not individual spots, so the severity numbers it produces
> are not meaningful yet. Use it to test the pipeline; after re-labelling one polygon per spot,
> set `VERSION` to the new version number and run again.

In [ ]:
!pip install -q ultralytics roboflow

## 1. Download the dataset
The API key is asked for at runtime so it never gets saved in the notebook.

In [ ]:
from getpass import getpass
from roboflow import Roboflow

WORKSPACE = "krishan-dev-s-workspace"
PROJECT = "coconut-diseases-spymp"
VERSION = 3

rf = Roboflow(api_key=getpass("Roboflow API key: "))
dataset = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download("yolov8")

In [ ]:
# Roboflow writes relative paths into data.yaml; Ultralytics needs absolute ones.
import yaml

DATA = f"{dataset.location}/data.yaml"
cfg = yaml.safe_load(open(DATA))
for key, split in [("train", "train"), ("val", "valid"), ("test", "test")]:
    cfg[key] = f"{dataset.location}/{split}/images"
yaml.safe_dump(cfg, open(DATA, "w"))
print("classes:", cfg["names"])

## 2. Train
`yolov8n-seg` is the smallest model (~7 MB), well inside the 60 MB on-device budget.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n-seg.pt")
model.train(data=DATA, imgsz=640, epochs=100, batch=8, patience=30, seed=0, project="runs", name="lesion_seg")
BEST = str(model.trainer.best)
print("best weights:", BEST)

## 3. Evaluate on the held-out test split

In [ ]:
best = YOLO(BEST)
metrics = best.val(data=DATA, split="test")
print(f"mask mAP50 = {metrics.seg.map50:.3f}   mask mAP50-95 = {metrics.seg.map:.3f}")

## 4. Severity (Table 3 rubric)

Leaf area is estimated from green–yellow pixels (plus the predicted lesions), so sky and soil
are excluded. Healthy = no lesion pixels.

In [ ]:
import cv2
import numpy as np

LABELS = ["Healthy", "Trace", "Mild", "Moderate", "Severe"]
LESION = next(i for i, n in best.names.items() if n == "lesion")

def leaf_mask(bgr):
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    return cv2.inRange(hsv, (20, 60, 40), (90, 255, 255)) > 0  # green-yellow tissue

def severity(path, conf=0.25):
    bgr = cv2.imread(path)
    h, w = bgr.shape[:2]
    r = best.predict(path, conf=conf, retina_masks=True, verbose=False)[0]
    lesion = np.zeros((h, w), bool)
    if r.masks is not None:
        for m, c in zip(r.masks.data.cpu().numpy(), r.boxes.cls.cpu().numpy()):
            if int(c) == LESION:
                lesion |= cv2.resize(m, (w, h), interpolation=cv2.INTER_NEAREST).astype(bool)
    leaf = leaf_mask(bgr) | lesion
    pct = 100 * lesion.sum() / max(leaf.sum(), 1)
    score = 0 if not lesion.any() else 1 + sum(pct > t for t in (5, 20, 50))
    return pct, score, r

In [ ]:
import glob, os
from PIL import Image
from IPython.display import display

test_images = sorted(glob.glob(f"{dataset.location}/test/images/*"))
for f in test_images:
    pct, score, _ = severity(f)
    print(f"{os.path.basename(f)[:32]:34s} lesion/leaf = {pct:5.1f} %  ->  {score} ({LABELS[score]})")

_, _, r = severity(test_images[0])
display(Image.fromarray(r.plot()[:, :, ::-1]))

## 5. Download all metrics as a zip

`lesion_seg_v<VERSION>_metrics.zip` contains:
- `test_metrics.json` – mask/box mAP, precision, recall on the test split
- `test_severity.csv` – lesion/leaf % and 0–4 severity per test image
- `test_eval/` – test confusion matrix, PR / F1 curves, prediction samples
- `train_run/` – `results.csv` (per-epoch losses and mAP), training curves, `args.yaml`

In [ ]:
import csv, json, shutil
from google.colab import files

out = "/content/metrics"
os.makedirs(out, exist_ok=True)

test = best.val(data=DATA, split="test", project=out, name="test_eval", exist_ok=True)
summary = {
    "dataset_version": VERSION,
    "mask_mAP50": float(test.seg.map50),
    "mask_mAP50_95": float(test.seg.map),
    "mask_precision": float(test.seg.mp),
    "mask_recall": float(test.seg.mr),
    "box_mAP50": float(test.box.map50),
    "box_mAP50_95": float(test.box.map),
}
json.dump(summary, open(f"{out}/test_metrics.json", "w"), indent=2)
print(json.dumps(summary, indent=2))

with open(f"{out}/test_severity.csv", "w", newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["image", "lesion_leaf_pct", "severity", "label"])
    for f in test_images:
        pct, score, _ = severity(f)
        w.writerow([os.path.basename(f), round(pct, 2), score, LABELS[score]])

run_dir = os.path.dirname(os.path.dirname(BEST))  # runs/lesion_seg
shutil.copytree(run_dir, f"{out}/train_run", dirs_exist_ok=True, ignore=shutil.ignore_patterns("*.pt"))

zip_path = shutil.make_archive(f"/content/lesion_seg_v{VERSION}_metrics", "zip", out)
files.download(zip_path)

## 6. Export for the app
ONNX runs in the browser with `onnxruntime-web`; keep `best.pt` for retraining.

In [ ]:
from google.colab import files

onnx_path = best.export(format="onnx", imgsz=640, simplify=True)
print("ONNX size: %.1f MB" % (os.path.getsize(onnx_path) / 1e6))
files.download(BEST)
files.download(onnx_path)